In [ ]:
## Aula 3: Do catálogo ao espectro de potência angular
# Isabela, Louis, Bruno

In [ ]:
# Importing python packages

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
import glass.shells
import glass.fields
import glass.points
import glass.galaxies
import glass.lensing
import glass.observations
import glass.ext.camb
from cosmology.compat.camb import Cosmology
import pymaster as nmt
import healpy as hp
import camb
import os


rcParams['font.size'] = 14
rcParams['axes.labelsize'] = 14
rcParams['xtick.labelsize'] = 14
rcParams['ytick.labelsize'] = 14

## 1. Gerando um catálogo de galáxias

Colocar aqui alguma explicacao sobre simulacoes lognormais

### 1.a) Calculando o espectro de potência angular da matéria

In [ ]:
# Onde salvar os arquivos de saída
storage = "../data/aula_3" # modificar para o caminho correto
matter_cls_dir = storage+'matter_cls'
os.makedirs(os.path.dirname(f'{matter_cls_dir}/cls.npy'), exist_ok=True)
    
# Parâmetros da simulação
lmax = 300
zmax = 1
dz = 0.2

# Parâmetros cosmológicos (Planck 2018)
H0 = 67.36
omch2 = 0.1225
ombh2 = 0.0223
As = 2.2e-9
ns = 0.96
tau = 0.06

pars = camb.set_params(H0=H0, omch2=omch2, ombh2=ombh2,
                        As=As, ns=ns, tau=tau,
                        NonLinear=camb.model.NonLinear_both)

# Cascas de redshift
zb = glass.shells.redshift_grid(0., zmax, dz=dz)

# Função janela (top-hat)
ws = glass.shells.tophat_windows(zb, dz=dz/100)

# Calculando o espectro de potência angular da matéria com CAMB
cls = glass.camb.matter_cls(pars, lmax, ws)
np.save(f'{matter_cls_dir}/cls', cls) # matriz com Nmultipoles*Nzbins*(Nzbins+1)/2 elementos


Adicionar aqui plot dos Cls da materia (talvez plot 3D?)

### 1.b) Criando galáxias

In [ ]:
# Calculando o espectro de potência do campo Gaussiano correspondente

# Resolução do mapa pixelizado
nside = 128
# Número de casacas de redshift cuja correlação será considerada (corr=None: todas as casacas são correlacionadas)
corr = None

# Número de bins de redshift e valor médio de cada bin
Nbins = len(zb)-1
zbar = (zb[1:]+zb[:-1])/2

# Load Cls da matéria
cls = np.load(matter_cls_dir+'/cls.npy')

# Definindo os campos lognormais
fields = glass.lognormal_fields(ws)

# Calculando os Cls Gaussianos    
Cls = glass.discretized_cls(cls, nside=nside, lmax=lmax, ncorr=corr)
gls = glass.solve_gaussian_spectra(fields, Cls)

# Salvando os Cls Gaussianos
os.makedirs(os.path.dirname(f'{matter_cls_dir}/gls.npy'), exist_ok=True)
arr = np.array(gls, dtype=object)
np.save(f'{matter_cls_dir}/gls', arr, allow_pickle=True)


Aqui podemos ter como exercicio calular a funcao de correlacao angular C(theta) a partir dos Cls e dos Gls para comparacao.

Tambem podemos adicionar propositalmente casos problematicos pra simulacao lognormal, onde algum gl tem diagonal negativa ou o lmax nao é grande o suficiente pra fazer C(theta) ser > -1. Podemos mostrar como contornar esses problemas. Talvez isso seja um pouco tecnico demais (??)

In [ ]:
# Definindo a função de viés de galáxias e a densidade de galáxias por arcmin²

arcmin2_sky = 148510800

def bias_g(z):
  """Galaxy bias"""
  return 0.971 + 1.042*z

def dndz_g(z):
  """Galaxy redshift distribution"""
  return 0.5 * z**2 * np.exp(-z/0.5) * arcmin2_sky

## 2. Obtendo um mapa pixelizado

## 3. Calculando o espectro de potência angular